# Figure S7 - comparing word forms

In [ ]:
from datetime import datetime
import pandas as pd
import numpy as np
import os
import time

import plotly.offline as pyo
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.templates.default = "plotly_white" # https://medium.com/plotly/introducing-plotly-py-theming-b644109ac9c7


#for fig rendering errors:
#plotly.offline.init_notebook_mode(connected=True)
#pio.renderers.default = 'iframe_connected' 
#from plotly.offline import init_notebook_mode, iplot
#init_notebook_mode(connected=True)
#pyo.init_notebook_mode()


# Load Storywrangler output csv version

In [ ]:
#LOAD ORIGNAL Dataset

#Originals
directory = r'C:\Users\markm\OneDrive\Akadeemiline\PhD RESEARCH\Artiklid\Ukraine-Twitter\Data\Final datasets used in article'
os.chdir(directory)

DForig = pd.read_csv(r'28languagesUA.csv', sep=',') #, index_col=[0,1], skipinitialspace=True)
DForig.date=pd.to_datetime(DForig.date)#,dayfirst=True

DForig = DForig.groupby(['language']).resample('W', on='date').mean(numeric_only=True).reset_index()

DForig

In [ ]:
#Load new larger all casesdataset
directory = r'C:\Users\markm\OneDrive\Akadeemiline\PhD RESEARCH\Artiklid\Ukraine-Twitter\Data\Final datasets used in article'
os.chdir(directory)
DFall = pd.read_excel(r'All_cases_allmetadata.xlsx')#, sep=',')
'example in English'
''' #if not already in weekly format
DFall = DFall[['query', 'language', 'date', 'count', 'freq']] # Ensure the necessary columns are selected
DFall['date'] = pd.to_datetime(DFall['date'], dayfirst=True)# Convert the 'date' column to datetime format
# Group by both 'query' and 'language', then resample weekly on 'date'
DFall = (
    DFall
    .groupby(['query', 'language']) #considers that some queries are shared between langauges
    .resample('W', on='date')
    .mean(numeric_only=True)  # Computes mean for 'count' and 'freq'
    .reset_index()
)
''' 
#First normalize then log
# Calculate the mean log_freq for each query within each language
mean_freq = DFall.groupby(['query', 'language'])['freq'].transform('mean')
# Normalize the log_freq by dividing by the mean
DFall['normalized_freq_log'] = np.log10(DFall['freq'] / mean_freq)

#First log then normalize
#DFall['log_freq'] = np.log10(DFall['freq'])
#mean_log_freq = DFall.groupby(['query', 'language'])['log_freq'].transform('mean')
#DFall['normalized_log_freq'] = DFall['log_freq'] / mean_log_freq

DFall


### Weekly freq sums

In [ ]:

# Assuming DFall is your DataFrame
# Group by 'language' and 'query', then sum the 'freq' values
DFsummed_freq = DFall.groupby(['language', 'query'])['freq'].sum().reset_index()

# Rename the 'freq' column to something more descriptive if needed
DFsummed_freq.rename(columns={'freq': 'total_freq'}, inplace=True)

# Assuming DFall is your DataFrame
# Group by 'language' and 'query', then sum the 'freq' values and take the first value for metadata columns
DFsummed_freq = DFall.groupby(['language', 'query']).agg(
    total_freq=('freq', 'sum'),
    type=('type', 'first'),
    plurality=('plurality', 'first'),
    gender=('gender', 'first'),
    case=('case', 'first'),
    demonym=('demonym', 'first'),
    attributes=('attributes', 'first'),
    marked_noun=('marked_noun', 'first')
).reset_index()

DFsummed_freq

In [ ]:
DFsorted_freq = DFsummed_freq.sort_values(by='total_freq', ascending=False)


### Filter samples for each language

In [ ]:
#NOUNS
#nr of samples
sampleNR=4

# Define a function to select X exemplary nouns per language, including the marked noun
def select_exemplary_nouns(group):
    # Start by selecting the marked noun if it exists
    marked = group[group['marked_noun'].notna()]
    
    # Priority 1: Select rows where 'type' is exactly "Noun"
    priority1 = group[(group['type'] == 'Noun') & (group['marked_noun'].isna())]
    
    # Priority 2: If less than 4, select rows where 'type' includes both "Noun" and "Adjective"
    if len(priority1) < sampleNR:
        priority2 = group[(group['type'].str.contains('Noun')) & (group['type'].str.contains('Adjective')) & (group['marked_noun'].isna())]
        priority1 = pd.concat([priority1, priority2])
    
    # Combine the marked noun with the selected exemplary nouns, limiting to 4 more (5 total)
    combined = pd.concat([marked, priority1.head(sampleNR)]).reset_index(drop=True)
    
    return combined

# Apply the selection criteria to each language group
DF_subset_nouns_unique = DFsummed_freq.groupby('language').apply(select_exemplary_nouns).reset_index(drop=True)

print(DF_subset_nouns_unique.language.value_counts())
#display(DF_subset_nouns_unique)

#Now take this subset from the data
DF_subset_nouns = DFall.merge(
    DF_subset_nouns_unique[['language', 'query']].drop_duplicates(),
    on=['language', 'query'],
    how='inner'
).reset_index(drop=True)
#display(DF_subset_nouns)


In [ ]:
#ADJECTIVES
# Number of samples to select per language
sampleNR = 4

# Define a function to select exemplary adjectives per language, including the marked adjective
def select_exemplary_adjectives(group):
    # Start by selecting the marked adjective if it exists
    marked = group[group['marked_noun'].notna()]
    
    # Priority 1: Select rows where 'type' is exactly "Adjective"
    priority1 = group[(group['type'] == 'Adjective') & (group['marked_noun'].isna())]
    
    # Priority 2: If less than sampleNR, select rows where 'type' includes both "Adjective" and another part of speech
    if len(priority1) < sampleNR:
        # Adjust the following line based on how combined types are represented in your 'type' column
        # For example, if combined types are like "Adjective and Noun", use appropriate string matching
        priority2 = group[(group['type'].str.contains('Adjective')) & 
                          (group['type'].str.contains('Noun')) & 
                          (group['marked_noun'].isna())]
        priority1 = pd.concat([priority1, priority2])
    
    # Combine the marked adjective with the selected exemplary adjectives, limiting to 'sampleNR' more
    combined = pd.concat([marked, priority1.head(sampleNR)]).reset_index(drop=True)
    
    return combined

# Apply the selection criteria to each language group
DF_subset_adjectives_unique = DFsummed_freq.groupby('language').apply(select_exemplary_adjectives).reset_index(drop=True)

# Display the count of adjectives per language in the subset
print(DF_subset_adjectives_unique.language.value_counts())

# Display the subset DataFrame
#display(DF_subset_adjectives_unique)

#Now take this subset from the data
DF_subset_adjectives = DFall.merge(
    DF_subset_adjectives_unique[['language', 'query']].drop_duplicates(),
    on=['language', 'query'],
    how='inner'
).reset_index(drop=True)
#display(DF_subset_adjectives)


In [ ]:
DF_subset_adjectives#.marked_noun.value_counts()

In [ ]:
# ================================================================
# COMBINE NOUNS + ADJECTIVES
# - weekly rows preserved
# - marked query only once per language
# - languages ordered by marked total freq
# - queries ordered: marked first, then freq (desc)
# ================================================================

import pandas as pd

# ------------------------------------------------
# 0. Combine weekly noun + adjective subsets
# ------------------------------------------------
DF_combined = pd.concat(
    [DF_subset_nouns, DF_subset_adjectives],
    ignore_index=True
)

# ------------------------------------------------
# 1. Build QUERY-LEVEL helper table
#    (one row per language–query)
# ------------------------------------------------
query_level = (
    DF_combined
    .groupby(['language', 'query'], as_index=False)
    .agg(
        total_freq=('freq', 'sum'),
        marked=('marked_noun', lambda x: x.notna().any())
    )
)

# ------------------------------------------------
# 2. Order LANGUAGES by marked query total freq
# ------------------------------------------------
language_order = (
    query_level[query_level['marked']]
    .groupby('language')['total_freq']
    .sum()
    .sort_values(ascending=False)
    .index
    .tolist()
)

# ------------------------------------------------
# 3. Order QUERIES within each language
#    marked first, then total freq (desc)
# ------------------------------------------------
query_level = query_level.sort_values(
    by=['language', 'marked', 'total_freq'],
    ascending=[True, False, False]
)

query_order = (
    query_level
    .groupby('language')['query']
    .apply(list)
    .to_dict()
)

# ------------------------------------------------
# 4. Create explicit integer sort keys
# ------------------------------------------------
lang_rank = {lang: i for i, lang in enumerate(language_order)}

query_rank = {
    lang: {q: i for i, q in enumerate(qs)}
    for lang, qs in query_order.items()
}

DF_combined['lang_rank'] = DF_combined['language'].map(lang_rank)
DF_combined['query_rank'] = DF_combined.apply(
    lambda r: query_rank[r['language']][r['query']],
    axis=1
)

# ------------------------------------------------
# 5. Final sort (language → query → time)
# ------------------------------------------------
DF_combined = DF_combined.sort_values(
    by=['lang_rank', 'query_rank', 'date']
).reset_index(drop=True)

# Cleanup
DF_combined = DF_combined.drop(columns=['lang_rank', 'query_rank'])

# ------------------------------------------------
# Result
# ------------------------------------------------
DF_combined


In [ ]:
# ================================================================
# 1. ADD LABELED QUERY COLUMN TO DF_combined
# ================================================================

def pos_label(t):
    t = str(t).lower()
    has_noun = 'noun' in t
    has_adj = 'adjective' in t
    
    if has_noun and has_adj:
        return 'N/A'
    elif has_noun:
        return 'N'
    elif has_adj:
        return 'A'
    else:
        return ''

DF_combined['query_pos'] = (
    DF_combined['query'] + ' (' + DF_combined['type'].apply(pos_label) + ')'
)

# --------------------------------------------------
# 2, Add query_pos to DF_summed_freq
# --------------------------------------------------
DFsummed_freq['query_pos'] = DFsummed_freq['query'] + ' (' + DFsummed_freq['type'].apply(
    lambda t: 'N/A' if 'noun' in t.lower() and 'adjective' in t.lower() 
              else ('N' if 'noun' in t.lower() 
                    else ('A' if 'adjective' in t.lower() else ''))
) + ')'

DF_combined.head(2)


In [ ]:
#DF_combined.pos_label.value_counts()

## Heatmaps (Functions)
failed to change barplot text size

#### Barplot names are always on the right of it (one didnt fit, needs manual fix)

In [ ]:
# Sort summed freq by 'total_freq' 
# Sort languages by total frequency
total_freq_by_language = DFsummed_freq.groupby('language')['total_freq'].sum()
sorted_languages = total_freq_by_language.sort_values(ascending=False).index.tolist()
#sorted_languages

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import pandas as pd

# Custom RdBu-inspired color scale
custom_colorscale = [
    [0, "#e3f3fc"],   # Lighter blue for underexpression
    [0.5, "#8E8E8E"], # Neutral
    [1, "#bd2f36"]    # Red for overexpression
]

zmin_value = DF_combined.normalized_freq_log.min()
zmax_value = DF_combined.normalized_freq_log.max()

def create_language_heatmap_with_totals(
    df_all,
    df_summed_freq,
    pixels_per_query=30,
    title="",
    plot_width=2126,
    plot_height=2126*3.9,
    subplot_title_font_size=32,
    text_font_size=32,
    sorting=sorted_languages  # list of languages in desired order
):
    df_all = df_all.copy()
    df_summed_freq = df_summed_freq.copy()

    # Use query_pos instead of query
    df_all['lang_query'] = df_all['language'] + ' - ' + df_all['query_pos']
    df_summed_freq['lang_query'] = df_summed_freq['language'] + ' - ' + df_summed_freq['query_pos']

    # Pivot data for heatmap
    pivot_freq = df_all.pivot_table(
        index=['language', 'date'],
        columns='lang_query',
        values='normalized_freq_log',
        aggfunc='sum',
        fill_value=0
    ).reset_index()

    # Sort queries by total frequency
    sorted_queries = df_summed_freq.sort_values(by='total_freq', ascending=True)['lang_query'].tolist()
    num_languages = len(sorting)

    # Calculate queries per language
    queries_per_language = []
    sorted_present_queries_dict = {}

    for lang in sorting:
        language_data = pivot_freq[pivot_freq['language'] == lang].copy().set_index('date')
        query_columns = [col for col in language_data.columns if col.startswith(f"{lang} - ")]
        language_data[query_columns] = language_data[query_columns].replace(0, np.nan)

        # Keep all queries present in data
        present_queries = [q for q in query_columns if language_data[q].notna().any()]
        sorted_present_queries = [q for q in sorted_queries if q in present_queries]
        sorted_present_queries_dict[lang] = sorted_present_queries
        queries_per_language.append(len(sorted_present_queries))

    # Normalize subplot heights
    per_language_height = [num * pixels_per_query for num in queries_per_language]
    row_heights_normalized = [h / sum(per_language_height) for h in per_language_height]

    # Create subplots (heatmap + total frequency bar)
    subplot_titles = []
    for lang, num_queries in zip(sorting, queries_per_language):
        subplot_titles.append(f"{lang.upper()}")
        subplot_titles.append("")  # bar plot has no title

    fig = make_subplots(
        rows=num_languages, cols=2,
        shared_xaxes=True,
        shared_yaxes=False,
        vertical_spacing=0.011,
        horizontal_spacing=0.01,
        column_widths=[0.85, 0.15],
        row_heights=row_heights_normalized,
        subplot_titles=subplot_titles
    )

    # Plot heatmaps and bar charts
    for i, lang in enumerate(sorting, start=1):
        sorted_present_queries = sorted_present_queries_dict[lang]
        language_data = pivot_freq[pivot_freq['language'] == lang].copy().set_index('date')
        language_data[sorted_present_queries] = language_data[sorted_present_queries].replace(0, np.nan)

        # Y-axis labels: highlight marked word
        y_labels = []
        for q in sorted_present_queries:
            # Check if this query is marked
            marked = df_summed_freq.loc[df_summed_freq['lang_query'] == q, 'marked_noun'].any()
            label = q.split(' - ', 1)[1]
            if marked:
                label = f"<b>{label}</b>"  # make marked word bold
            y_labels.append(label)

        # --- Heatmap ---
        heatmap = go.Heatmap(
            z=language_data[sorted_present_queries].T,
            x=language_data.index,
            y=y_labels,
            colorscale=custom_colorscale,
            zmin=zmin_value,
            zmid=0,
            zmax=zmax_value,
            showscale=True,
            colorbar=dict(
                orientation='h',
                x=0.4,
                y=-0.04,
                thickness=60,
                len=0.5,
                tickfont=dict(size=24)
            )
        )
        fig.add_trace(heatmap, row=i, col=1)

        
        fig.update_yaxes(
            tickmode='array',
            tickvals=y_labels,
            ticktext=y_labels,
            row=i,
            col=1
        )

        # --- Bar chart for total frequency ---
        total_freq_for_lang = df_summed_freq[df_summed_freq['language'] == lang].set_index('lang_query')
        total_freq_for_lang = total_freq_for_lang.loc[sorted_present_queries, 'total_freq']
        total_freq_for_lang = total_freq_for_lang.reindex(sorted_present_queries)

        bar_chart = go.Bar(
            x=total_freq_for_lang,
            y=y_labels,
            orientation='h',
            marker=dict(color='rgba(50, 171, 96, 0.6)'),
            showlegend=False,
            text=total_freq_for_lang.apply(lambda x: f"{x:.4f}"),
            textposition='outside',
            hoverinfo='x+y',
            textfont=dict(size=text_font_size, color='black')
        )
        fig.add_trace(bar_chart, row=i, col=2)

    # Layout
    fig.update_layout(
        width=plot_width,
        height=plot_height,
        title_text=title,
        showlegend=False,
        margin=dict(l=20, r=20, t=50, b=40)
    )

    # Subplot titles font size
    for annotation in fig['layout']['annotations']:
        annotation['font']['size'] = subplot_title_font_size

    # Axes formatting
    for i in range(1, num_languages + 1):
        if i != num_languages:
            fig.update_xaxes(showticklabels=False, row=i, col=1)
            fig.update_xaxes(showticklabels=False, row=i, col=2)
        else:
            fig.update_xaxes(
                title_text="",
                row=i,
                col=1,
                tickfont=dict(size=24)
            )
            fig.update_xaxes(
                title_text="Relative frequency sum",
                row=i,
                col=2,
                tickfont=dict(size=20)
            )
        fig.update_yaxes(tickfont=dict(size=text_font_size), row=i, col=1)
        fig.update_yaxes(showticklabels=False, row=i, col=2)

    fig.show()
    return fig
fig= create_language_heatmap_with_totals(DF_combined, DFsummed_freq)


# Save

In [ ]:
# OLD SAVING METHOD — Requires old Plotly version to render correctly!

import os
from subprocess import call
plot_width = 2126    # Define the width of the plot
plot_height = 2126*3.9 # Calculate height based on aspect ratio

# Set the figure directory and filename criteria
sublocation = '/Visualizations/Fig.S7_Different word forms comparison/Together nouns and adjectives/correct/'  # Define the sublocation for saving files

# Define the plot name
name = 'Wordform_comparison'  # Replace with your desired plot name
#name = 'Wordform_comparison_ADJ'  # Replace with your desired plot name

# Define the base directory for saving files
directory = r'C:\Users\markm\OneDrive\Akadeemiline\PhD RESEARCH\Artiklid\Ukraine-Twitter'
os.chdir(directory)

# Make the main figure directory if not present
main_figure_path = directory + sublocation
os.makedirs(main_figure_path, exist_ok=True)

# Define output formats and create their subdirectories
formats = ['pdf', 'jpeg', 'svg']
for fmt in formats:
    os.makedirs(main_figure_path + fmt + "/", exist_ok=True)  # Use forward slash for consistency

# Save the figure in different formats
for fmt in formats:
    file_path = main_figure_path + fmt + "/" + name + "." + fmt  # Ensure consistent path formatting
    fig.write_image(file_path, 
                    format=fmt,
                    width=plot_width,
                    height=plot_height,
    )#, engine="kaleido"
    print(f"Saved {fmt.upper()} to: {file_path}")

# Convert PDF to EPS if needed
# call(["pdf2ps", main_figure_path + "pdf" + "/" + "_" + name + ".pdf", main_figure_path + "eps" + "/" + "_" + name + ".eps"])
